# Experiment B v4 colabpaths: zonal OCR + QR cascade + dedupe

Quality run for end-to-end CSV fields. This is the first Colab candidate for metric improvement.

Runtime: Google Colab GPU. Default guard requires one T4 GPU.
If Colab gives L4/A100 instead, change `EXP_REQUIRE_GPU_NAME`.

This notebook uses only local CV/OCR libraries inside Colab. It does not call cloud OCR/API/LLM during solution inference.

In [ ]:
!nvidia-smi
import sys
print(sys.version)

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

DATASET = "whitenigger/lenta-shelf-ai-bundle"
BUNDLE_DIR = Path("/content/lenta_bundle")
KAGGLE_INPUT = Path("/content/kaggle/input/lenta-shelf-ai-bundle")
KAGGLE_WORKING = Path("/content/kaggle/working")
RUN_TS = time.strftime("%Y%m%d_%H%M%S")
DRIVE_ROOT = Path("/content/drive/MyDrive/lenta_colab_runs")


def run(cmd, cwd=None, env=None, check=True):
    print("[RUN]", " ".join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, check=check)


def setup_drive():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        return True
    except Exception as exc:
        print("[WARN] Drive is not mounted:", repr(exc))
        return False


def setup_kaggle_credentials():
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    target = kaggle_dir / "kaggle.json"

    username = os.environ.get("KAGGLE_USERNAME")
    key = os.environ.get("KAGGLE_KEY")
    if not (username and key):
        try:
            from google.colab import userdata
            username = username or userdata.get("KAGGLE_USERNAME")
            key = key or userdata.get("KAGGLE_KEY")
        except Exception:
            pass
    if username and key:
        target.write_text(json.dumps({"username": username, "key": key}), encoding="utf-8")
        target.chmod(0o600)
        print("[OK] Kaggle credentials loaded from env/Colab secrets")
        return

    candidates = list(Path("/content").glob("kaggle*.json"))
    if candidates:
        shutil.copy2(candidates[0], target)
        target.chmod(0o600)
        print("[OK] Kaggle credentials copied from", candidates[0])
        return

    from google.colab import files
    print("Upload kaggle.json now. The file stays inside this Colab runtime.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("kaggle.json was not uploaded")
    src = Path("/content") / next(iter(uploaded.keys()))
    shutil.copy2(src, target)
    target.chmod(0o600)
    print("[OK] Kaggle credentials uploaded")


def prepare_bundle():
    run([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
    setup_kaggle_credentials()
    BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
    if not (BUNDLE_DIR / "repo.zip").exists() and not (BUNDLE_DIR / "repo").exists():
        run(["kaggle", "datasets", "download", "-d", DATASET, "-p", str(BUNDLE_DIR), "--unzip"])

    print("[BUNDLE_FILES]", [str(p.relative_to(BUNDLE_DIR)) for p in sorted(BUNDLE_DIR.glob("**/*"))[:80]])
    repo_zip = next(iter(BUNDLE_DIR.glob("**/repo.zip")), None)
    data_zip = next(iter(BUNDLE_DIR.glob("**/data.zip")), None)
    repo_dir = next((p for p in BUNDLE_DIR.glob("**/repo") if p.is_dir()), None)
    data_dir = next((p for p in BUNDLE_DIR.glob("**/data") if p.is_dir()), None)
    if repo_dir is None:
        repo_candidates = [p.parent for p in BUNDLE_DIR.glob("**/kaggle/gpu_experiment.py")]
        repo_dir = repo_candidates[0] if repo_candidates else None
    if data_dir is None:
        data_candidates = [p.parent.parent for p in BUNDLE_DIR.glob("**/*/*.mp4") if "repo" not in p.parts]
        data_dir = data_candidates[0] if data_candidates else None
    if repo_zip is None and repo_dir is None:
        raise FileNotFoundError(f"Cannot find repo.zip or repo folder in {BUNDLE_DIR}")
    if data_zip is None and data_dir is None:
        raise FileNotFoundError(f"Cannot find data.zip or data folder in {BUNDLE_DIR}")

    KAGGLE_INPUT.mkdir(parents=True, exist_ok=True)
    KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
    if repo_zip is not None:
        shutil.copy2(repo_zip, KAGGLE_INPUT / "repo.zip")
    else:
        shutil.copytree(repo_dir, KAGGLE_INPUT / "repo", dirs_exist_ok=True)
    if data_zip is not None:
        shutil.copy2(data_zip, KAGGLE_INPUT / "data.zip")
    else:
        shutil.copytree(data_dir, KAGGLE_INPUT / "data", dirs_exist_ok=True)

    runner_root = Path("/content/lenta_runner_repo")
    if runner_root.exists():
        shutil.rmtree(runner_root)
    runner_root.mkdir(parents=True)
    if repo_zip is not None:
        with zipfile.ZipFile(repo_zip) as zf:
            zf.extractall(runner_root)
        script = runner_root / "repo" / "kaggle" / "gpu_experiment.py"
    else:
        shutil.copytree(repo_dir, runner_root / "repo", dirs_exist_ok=True)
        script = runner_root / "repo" / "kaggle" / "gpu_experiment.py"
    if not script.exists():
        raise FileNotFoundError(script)
    # The runner is shared with Kaggle and hardcodes /kaggle/input and
    # /kaggle/working. In Colab those paths can be read-only or absent, so run a
    # Colab-local patched copy while keeping the repository source unchanged.
    source = script.read_text(encoding="utf-8")
    source = source.replace('Path("/kaggle/input")', 'Path("/content/kaggle/input")')
    source = source.replace('Path("/kaggle/working/artifacts")', 'Path("/content/kaggle/working/artifacts")')
    script.write_text(source, encoding="utf-8")
    print("[OK] runner:", script)
    return script


def run_experiment(script, exp_env):
    artifacts = KAGGLE_WORKING / "artifacts"
    if artifacts.exists():
        shutil.rmtree(artifacts)
    input_bundle = KAGGLE_WORKING / "input_bundle"
    if input_bundle.exists():
        shutil.rmtree(input_bundle)

    env = os.environ.copy()
    env.update({k: str(v) for k, v in exp_env.items()})
    run([sys.executable, str(script)], env=env)
    return artifacts


def run_error_analysis():
    work = Path("/tmp/lenta_shelf_ai_solution")
    eval_dir = work / "outputs" / "eval_public_fast"
    bundle = KAGGLE_WORKING / "input_bundle"
    if not eval_dir.exists() or not bundle.exists():
        print("[WARN] no eval outputs or input bundle for error analysis")
        return
    out_dir = KAGGLE_WORKING / "artifacts" / "error_analysis"
    out_dir.mkdir(parents=True, exist_ok=True)
    gt_by_stem = {p.stem: p for p in bundle.glob("data/**/*/*.csv")}
    for pred_csv in sorted(eval_dir.glob("*_recognized.csv")):
        stem = pred_csv.name.replace("_recognized.csv", "")
        gt_csv = gt_by_stem.get(stem)
        if gt_csv is None:
            print("[WARN] no GT csv for", pred_csv)
            continue
        run([
            sys.executable,
            "scripts/analyze_final_errors.py",
            "--gt-csv",
            str(gt_csv),
            "--pred-csv",
            str(pred_csv),
            "--out-json",
            str(out_dir / f"{stem}_error_analysis.json"),
        ], cwd=work)


def save_artifacts(artifacts, experiment_name):
    run_dir = DRIVE_ROOT / f"{experiment_name}_{RUN_TS}"
    if DRIVE_ROOT.exists():
        if run_dir.exists():
            shutil.rmtree(run_dir)
        shutil.copytree(artifacts, run_dir / "artifacts")
        archive = shutil.make_archive(str(run_dir), "zip", run_dir)
        print("[OK] saved to Drive:", run_dir)
        print("[OK] zip:", archive)
        return run_dir
    local_dir = Path("/content") / f"{experiment_name}_{RUN_TS}"
    if local_dir.exists():
        shutil.rmtree(local_dir)
    shutil.copytree(artifacts, local_dir / "artifacts")
    archive = shutil.make_archive(str(local_dir), "zip", local_dir)
    print("[OK] saved locally:", local_dir)
    print("[OK] zip:", archive)
    return local_dir


def print_key_reports(artifacts):
    paths = [
        artifacts / "run_summary.json",
        artifacts / "kaggle_metrics" / "detection_recall_yolo_only.json",
        artifacts / "kaggle_metrics" / "detection_recall_hybrid.json",
        artifacts / "eval_public_fast" / "metrics.json",
    ]
    for path in paths:
        if not path.exists():
            print("[MISS]", path)
            continue
        print("\n" + "=" * 90)
        print(path)
        print("=" * 90)
        print(path.read_text(encoding="utf-8")[:12000])

In [ ]:
EXP_ENV = {
    "EXP_NAME": "exp-colab-b-zonal-qr-ocr-dedupe",
    "EXP_REQUIRE_GPU_NAME": "T4",
    "EXP_REQUIRE_GPU_COUNT": "1",
    "EXP_DEVICE": "0",
    "EXP_MODEL": "models/price_tag_yolo.pt",
    "EXP_REQUIREMENTS": "requirements-full.txt",
    "EXP_SKIP_TORCH_PIN": "1",
    "EXP_SKIP_TRAIN": "1",
    "EXP_SKIP_PUBLIC_EVAL": "0",
    "EXP_RECALL_CONF": "0.12",
    "EXP_REQUIRE_PADDLE": "1",
    "EXP_PIPELINE_ENABLE_OCR": "1",
    "EXP_PIPELINE_ENABLE_QR": "1",
    "EXP_PIPELINE_DEFER_OCR": "1",
    "EXP_PIPELINE_PREFER_PADDLE": "1",
    "EXP_PIPELINE_USE_GPU": "0",
    "EXP_PIPELINE_ENABLE_FALLBACKS": "0",
    "EXP_PIPELINE_YOLO_CONF": "0.12",
    "EXP_PIPELINE_SAMPLE_FPS": "4.0",
    "EXP_PIPELINE_MAX_DETECTIONS": "60",
    "EXP_PIPELINE_ENABLE_ZONAL_OCR": "1",
    "EXP_PIPELINE_QR_EXPANSION_X": "0.55",
    "EXP_PIPELINE_QR_EXPANSION_Y": "0.45",
    "EXP_PIPELINE_FALLBACK_REQUIRE_EVIDENCE": "1",
    "EXP_PIPELINE_DEDUPE_EXTENDED_WINDOW_MS": "12000",
    "EXP_PIPELINE_DEDUPE_VISUAL_HASH": "14",
    "EXP_PIPELINE_DEDUPE_TEXT_SIMILARITY": "0.86"
}
EXPERIMENT_NAME = EXP_ENV["EXP_NAME"]

setup_drive()
script = prepare_bundle()
artifacts = run_experiment(script, EXP_ENV)
run_error_analysis()
run_dir = save_artifacts(artifacts, EXPERIMENT_NAME)
print_key_reports(artifacts)

## Required critique after run

If QR/barcode/price fill does not improve, the next fix is crop rectification and QR-zone localization, not detector training.

Minimum evidence to call this useful:
- `eval_public_fast/metrics.json` has nonzero `good_rows_at_80`;
- `26_12-20 pred_rows` moves closer to GT without collapsing `matched_rows`;
- `field_fill` in error-analysis improves for prices, barcode, QR, SKU;
- no fallback spam returns.